# Phase 7 
Notebook will contain
- Imports and paths
- Environment/setup
- Load chunks
- Load existing FAISS index
- Load BGE embedding model
- Load cross-encoder
- Create reusable retriever
- Load your LLM
- Define the RAG prompt
- Run short development queries
- Run long development queries
- Save generated answers + retrieved context
- Evaluate answer quality
- Analyze retrieval-vs-generation errors
- Run untouched holdout evaluation
- Produce final Phase 7 metrics
- Save all results for your final research report

## Cell 1 — Install dependencies

In [2]:
# ============================================================
# PHASE 7 — END-TO-END RAG
# CELL 1: INSTALL DEPENDENCIES
# ============================================================

!pip -q install sentence-transformers faiss-cpu pandas numpy tqdm scikit-learn

## Cell 2 — Imports

In [3]:
# ============================================================
# CELL 2: IMPORTS
# ============================================================

import os
import sys
import json
import time
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import faiss

from tqdm.auto import tqdm
from sentence_transformers import SentenceTransformer, CrossEncoder

e:\Anaconda3\envs\voiceenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Cell 3 — Project paths

Use the same paths you used in Phase 6.5.

In [4]:
# ============================================================
# CELL 3: PROJECT PATHS
# ============================================================

BASE_DIR = Path(".")

CHUNKS_PATH = BASE_DIR / "chunked_data" / "chunks.jsonl"
FAISS_INDEX_PATH = BASE_DIR / "chunked_data" / "faiss_index.bin"

RAG_DIR = BASE_DIR / "rag"
OUTPUT_DIR = BASE_DIR / "phase7_outputs"

RAG_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("BASE_DIR:", BASE_DIR)
print("CHUNKS_PATH:", CHUNKS_PATH)
print("FAISS_INDEX_PATH:", FAISS_INDEX_PATH)
print("RAG_DIR:", RAG_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)

BASE_DIR: .
CHUNKS_PATH: chunked_data\chunks.jsonl
FAISS_INDEX_PATH: chunked_data\faiss_index.bin
RAG_DIR: rag
OUTPUT_DIR: phase7_outputs


## Cell 4 — Verify existing files

Do not create a new FAISS index here.

In [5]:
# ============================================================
# CELL 4: VERIFY EXISTING RETRIEVAL FILES
# ============================================================

assert CHUNKS_PATH.exists(), f"Missing chunks file: {CHUNKS_PATH}"
assert FAISS_INDEX_PATH.exists(), f"Missing FAISS index: {FAISS_INDEX_PATH}"

print("✓ chunks.jsonl found")
print("✓ faiss_index.bin found")

print("\nChunks file size:",
      round(CHUNKS_PATH.stat().st_size / 1024, 2), "KB")

print("FAISS index size:",
      round(FAISS_INDEX_PATH.stat().st_size / 1024, 2), "KB")

✓ chunks.jsonl found
✓ faiss_index.bin found

Chunks file size: 98.42 KB
FAISS index size: 144.04 KB


## Cell 5 — Frozen Phase 6.5 configuration

In [6]:
# ============================================================
# CELL 5: FROZEN PHASE 6.5 CONFIGURATION
# ============================================================

SEMANTIC_TOP_K = 10
FINAL_TOP_K = 5

EMBEDDING_MODEL_NAME = "BAAI/bge-small-en-v1.5"
RERANKER_MODEL_NAME = "cross-encoder/ms-marco-MiniLM-L-6-v2"

print("Frozen retrieval configuration")
print("--------------------------------")
print("Embedding model :", EMBEDDING_MODEL_NAME)
print("Reranker model  :", RERANKER_MODEL_NAME)
print("Semantic Top-K  :", SEMANTIC_TOP_K)
print("Final Top-K     :", FINAL_TOP_K)

Frozen retrieval configuration
--------------------------------
Embedding model : BAAI/bge-small-en-v1.5
Reranker model  : cross-encoder/ms-marco-MiniLM-L-6-v2
Semantic Top-K  : 10
Final Top-K     : 5


Cell 6 — Load chunks

In [7]:
# ============================================================
# CELL 6: LOAD CHUNKS
# ============================================================

chunks = []

with open(CHUNKS_PATH, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            chunks.append(json.loads(line))

print("Number of chunks:", len(chunks))

assert len(chunks) == 96, (
    f"Expected 96 clean corpus chunks, found {len(chunks)}"
)

print("\nExample chunk:")
print(json.dumps(chunks[0], indent=2, ensure_ascii=False)[:2000])

Number of chunks: 96

Example chunk:
{
  "chunk_id": "chunk_000001",
  "document": "Continuing Education.md",
  "file_path": "Benefits and Perks\\Continuing Education.md",
  "category": "Benefits and Perks",
  "section_path": [
    "Continuing Education"
  ],
  "section_title": "Continuing Education",
  "chunk_index": 1,
  "word_count": 43,
  "character_count": 243,
  "content": "One of Clef’s core values is “Be better today than yesterday,” so it’s important that we support our employees’ efforts to learn, grow, and improve. These are some of the key benefits of working at Clef, and are central to our company culture."
}


Cell 7 — Load FAISS index

In [8]:
# ============================================================
# CELL 7: LOAD EXISTING FAISS INDEX
# ============================================================

index = faiss.read_index(str(FAISS_INDEX_PATH))

print("FAISS index loaded successfully")
print("--------------------------------")
print("Number of vectors :", index.ntotal)
print("Vector dimension  :", index.d)

assert index.ntotal == len(chunks), (
    f"FAISS contains {index.ntotal} vectors, "
    f"but chunks.jsonl contains {len(chunks)} chunks."
)

print("\n✓ FAISS vector count matches chunk count")

FAISS index loaded successfully
--------------------------------
Number of vectors : 96
Vector dimension  : 384

✓ FAISS vector count matches chunk count


Cell 8 — Load embedding model

In [10]:
# ============================================================
# CELL 8: LOAD BGE EMBEDDING MODEL
# ============================================================

embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)

print("Embedding model loaded:")
print(EMBEDDING_MODEL_NAME)

print("Embedding dimension:",
      embedding_model.get_embedding_dimension())

assert (
    embedding_model.get_embedding_dimension()
    == index.d
), "Embedding dimension does not match FAISS index dimension."

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3395.01it/s]


Embedding model loaded:
BAAI/bge-small-en-v1.5
Embedding dimension: 384


Cell 9 — Load cross-encoder

In [11]:
# ============================================================
# CELL 9: LOAD CROSS-ENCODER RERANKER
# ============================================================

reranker = CrossEncoder(RERANKER_MODEL_NAME)

print("Cross-encoder loaded:")
print(RERANKER_MODEL_NAME)

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 3580.30it/s]


Cross-encoder loaded:
cross-encoder/ms-marco-MiniLM-L-6-v2


Cell 10 — Create config.py

In [12]:
# ============================================================
# CELL 10: CREATE rag/config.py
# ============================================================

config_code = r'''
SEMANTIC_TOP_K = 10
FINAL_TOP_K = 5

EMBEDDING_MODEL_NAME = "BAAI/bge-small-en-v1.5"
RERANKER_MODEL_NAME = "cross-encoder/ms-marco-MiniLM-L-6-v2"
'''

(RAG_DIR / "config.py").write_text(
    config_code,
    encoding="utf-8"
)

print("Created:", RAG_DIR / "config.py")

Created: rag\config.py


Cell 11 — Create __init__.py

In [13]:
# ============================================================
# CELL 11: CREATE rag/__init__.py
# ============================================================

init_code = r'''
from .retriever import RAGRetriever
'''

(RAG_DIR / "__init__.py").write_text(
    init_code,
    encoding="utf-8"
)

print("Created:", RAG_DIR / "__init__.py")

Created: rag\__init__.py


Cell 12 — Create retriever.py

This is the important part.

In [14]:
# ============================================================
# CELL 12: CREATE rag/retriever.py
# ============================================================

retriever_code = r'''
import json
from pathlib import Path

import faiss
import numpy as np
from sentence_transformers import SentenceTransformer, CrossEncoder

from .config import (
    SEMANTIC_TOP_K,
    FINAL_TOP_K,
    EMBEDDING_MODEL_NAME,
    RERANKER_MODEL_NAME,
)


class RAGRetriever:
    """
    Reusable Phase 6.5 retrieval pipeline.

    Pipeline:
        Query
          ↓
        BGE embedding
          ↓
        FAISS semantic retrieval
          ↓
        Top-10 candidates
          ↓
        Cross-Encoder reranking
          ↓
        Top-5 final chunks
    """

    def __init__(
        self,
        chunks_path,
        index_path,
        semantic_top_k=SEMANTIC_TOP_K,
        final_top_k=FINAL_TOP_K,
        embedding_model_name=EMBEDDING_MODEL_NAME,
        reranker_model_name=RERANKER_MODEL_NAME,
    ):

        self.chunks_path = Path(chunks_path)
        self.index_path = Path(index_path)

        self.semantic_top_k = semantic_top_k
        self.final_top_k = final_top_k

        # ----------------------------------------------------
        # Load chunks
        # ----------------------------------------------------

        if not self.chunks_path.exists():
            raise FileNotFoundError(
                f"Chunks file not found: {self.chunks_path}"
            )

        self.chunks = []

        with open(self.chunks_path, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()

                if line:
                    self.chunks.append(json.loads(line))

        # ----------------------------------------------------
        # Load FAISS index
        # ----------------------------------------------------

        if not self.index_path.exists():
            raise FileNotFoundError(
                f"FAISS index not found: {self.index_path}"
            )

        self.index = faiss.read_index(str(self.index_path))

        if self.index.ntotal != len(self.chunks):
            raise ValueError(
                "FAISS/chunk mismatch: "
                f"{self.index.ntotal} vectors vs "
                f"{len(self.chunks)} chunks."
            )

        # ----------------------------------------------------
        # Load embedding model
        # ----------------------------------------------------

        self.embedding_model = SentenceTransformer(
            embedding_model_name
        )

        embedding_dim = (
            self.embedding_model
            .get_sentence_embedding_dimension()
        )

        if embedding_dim != self.index.d:
            raise ValueError(
                "Embedding dimension mismatch: "
                f"model={embedding_dim}, "
                f"FAISS={self.index.d}"
            )

        # ----------------------------------------------------
        # Load cross-encoder
        # ----------------------------------------------------

        self.reranker = CrossEncoder(
            reranker_model_name
        )

    # ========================================================
    # Internal helper
    # ========================================================

    @staticmethod
    def _extract_text(chunk):
        """
        Extract the text field while keeping compatibility
        with common chunk JSON structures.
        """

        if isinstance(chunk, str):
            return chunk

        for key in [
            "text",
            "chunk_text",
            "content",
            "page_content",
        ]:
            if key in chunk:
                return str(chunk[key])

        raise KeyError(
            "Could not find chunk text field. "
            f"Available keys: {list(chunk.keys())}"
        )

    # ========================================================
    # Retrieve
    # ========================================================

    def retrieve(self, query):
        """
        Run complete semantic + reranker retrieval.

        Returns a list of dictionaries sorted by final
        cross-encoder score.
        """

        if not isinstance(query, str) or not query.strip():
            raise ValueError("Query must be a non-empty string.")

        query = query.strip()

        # ----------------------------------------------------
        # 1. Query embedding
        # ----------------------------------------------------

        query_embedding = self.embedding_model.encode(
            [query],
            normalize_embeddings=True,
            convert_to_numpy=True,
        ).astype("float32")

        # ----------------------------------------------------
        # 2. FAISS semantic retrieval
        # ----------------------------------------------------

        semantic_k = min(
            self.semantic_top_k,
            self.index.ntotal
        )

        distances, indices = self.index.search(
            query_embedding,
            semantic_k
        )

        candidate_indices = indices[0]
        candidate_distances = distances[0]

        candidates = []

        for rank, (idx, distance) in enumerate(
            zip(candidate_indices, candidate_distances),
            start=1
        ):

            idx = int(idx)

            if idx < 0 or idx >= len(self.chunks):
                continue

            chunk = self.chunks[idx]

            candidates.append({
                "chunk_index": idx,
                "semantic_rank": rank,
                "semantic_score": float(distance),
                "text": self._extract_text(chunk),
                "chunk": chunk,
            })

        if not candidates:
            return []

        # ----------------------------------------------------
        # 3. Cross-encoder reranking
        # ----------------------------------------------------

        pairs = [
            (query, candidate["text"])
            for candidate in candidates
        ]

        reranker_scores = self.reranker.predict(
            pairs,
            show_progress_bar=False
        )

        for candidate, score in zip(
            candidates,
            reranker_scores
        ):
            candidate["reranker_score"] = float(score)

        # ----------------------------------------------------
        # 4. Sort by cross-encoder score
        # ----------------------------------------------------

        candidates.sort(
            key=lambda x: x["reranker_score"],
            reverse=True
        )

        # ----------------------------------------------------
        # 5. Assign final rank
        # ----------------------------------------------------

        final_results = candidates[
            :min(self.final_top_k, len(candidates))
        ]

        for rank, result in enumerate(
            final_results,
            start=1
        ):
            result["final_rank"] = rank

        return final_results

    # ========================================================
    # Context helper
    # ========================================================

    def build_context(self, results):
        """
        Convert retrieved chunks into an ordered context
        string suitable for an LLM prompt.
        """

        context_parts = []

        for result in results:

            context_parts.append(
                f"[Chunk {result['final_rank']}]\n"
                f"{result['text']}"
            )

        return "\n\n".join(context_parts)
'''

(RAG_DIR / "retriever.py").write_text(
    retriever_code,
    encoding="utf-8"
)

print("Created:", RAG_DIR / "retriever.py")

Created: rag\retriever.py


Cell 13 — Reload Python module

In [15]:
# ============================================================
# CELL 13: IMPORT REUSABLE RETRIEVER
# ============================================================

if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))

from rag.retriever import RAGRetriever

print("✓ RAGRetriever imported successfully")

✓ RAGRetriever imported successfully


Cell 14 — Instantiate retriever

In [16]:
# ============================================================
# CELL 14: INITIALIZE FINAL RETRIEVER
# ============================================================

retriever = RAGRetriever(
    chunks_path=CHUNKS_PATH,
    index_path=FAISS_INDEX_PATH,
)

print("Retriever initialized successfully")
print("----------------------------------")
print("Semantic Top-K:", retriever.semantic_top_k)
print("Final Top-K   :", retriever.final_top_k)
print("Chunks        :", len(retriever.chunks))
print("FAISS vectors :", retriever.index.ntotal)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2411.15it/s]
e:\Projects\Speech AI\Data\rag\retriever.py:97: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  .get_sentence_embedding_dimension()
Loading weights: 100%|██████████| 105/105 [00:00<00:00, 3743.55it/s]


Retriever initialized successfully
----------------------------------
Semantic Top-K: 10
Final Top-K   : 5
Chunks        : 96
FAISS vectors : 96


  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.


Validate the reusable module

This step is important. We want to make sure modularization hasn't changed your Phase 6.5 retrieval behavior.

Cell 15 — Test one query

In [18]:
# ============================================================
# CELL 15: SINGLE QUERY RETRIEVAL TEST
# ============================================================

test_query = (
    "Is there work from home opportunity?"
)

results = retriever.retrieve(test_query)

print("Query:")
print(test_query)

print("\nRetrieved results:")
print("=" * 80)

for result in results:

    print(
        f"\nFinal Rank     : {result['final_rank']}"
        f"\nSemantic Rank  : {result['semantic_rank']}"
        f"\nSemantic Score : {result['semantic_score']:.4f}"
        f"\nReranker Score : {result['reranker_score']:.4f}"
        f"\nChunk Index    : {result['chunk_index']}"
    )

    print("\nText:")
    print(result["text"][:1000])
    print("-" * 80)

Query:
Is there work from home opportunity?

Retrieved results:

Final Rank     : 1
Semantic Rank  : 1
Semantic Score : 0.6869
Reranker Score : -1.9976
Chunk Index    : 50

Text:
If you're working remotely or from home, you should put a calendar event indicating where you are working from for all the time you are out of the office.
--------------------------------------------------------------------------------

Final Rank     : 2
Semantic Rank  : 5
Semantic Score : 0.6578
Reranker Score : -2.0388
Chunk Index    : 35

Text:
This policy is written in the context of our current company setup, which is that every employee works in the same office for the most of the work week. Specifically, this document addresses working from home on a regular basis and/or working remotely for up to one week out of any given month. It does not address:

* A fully remote employee
* An employee who works from the office 1wk/month or is remote for the majority of their time
* A team that is distributed

If 

Cell 16 — Verify Top-K behavior

In [19]:
# ============================================================
# CELL 16: VERIFY FINAL TOP-K
# ============================================================

assert len(results) <= FINAL_TOP_K

assert all(
    result["final_rank"] == i
    for i, result in enumerate(results, start=1)
)

# Verify reranker ordering
scores = [
    result["reranker_score"]
    for result in results
]

assert scores == sorted(
    scores,
    reverse=True
)

print("✓ Final result count:", len(results))
print("✓ Final ranks are correct")
print("✓ Results are sorted by reranker score")
print("✓ Top-5 reranked retrieval is working")

✓ Final result count: 5
✓ Final ranks are correct
✓ Results are sorted by reranker score
✓ Top-5 reranked retrieval is working


Cell 17 — Test context construction

In [20]:
# ============================================================
# CELL 17: TEST CONTEXT CONSTRUCTION
# ============================================================

context = retriever.build_context(results)

print(context[:5000])

[Chunk 1]
If you're working remotely or from home, you should put a calendar event indicating where you are working from for all the time you are out of the office.

[Chunk 2]
This policy is written in the context of our current company setup, which is that every employee works in the same office for the most of the work week. Specifically, this document addresses working from home on a regular basis and/or working remotely for up to one week out of any given month. It does not address:

* A fully remote employee
* An employee who works from the office 1wk/month or is remote for the majority of their time
* A team that is distributed

If and when the current company setup changes, we should address the above situations. For the purpose of this document, we interpret remote work as as a co-located employee of Clef working from home or from somewhere irregular.

[Chunk 3]
***An extended remote work period includes anything longer than 2 days or any period of time where we work from somew

Cell 18 — Save the module configuration

This creates a record of exactly what Phase 7 is using.

In [21]:
# ============================================================
# CELL 18: SAVE PHASE 7 RETRIEVAL CONFIGURATION
# ============================================================

retrieval_config = {
    "architecture": "Semantic Retrieval + Cross-Encoder Reranker",
    "embedding_model": EMBEDDING_MODEL_NAME,
    "reranker_model": RERANKER_MODEL_NAME,
    "semantic_top_k": SEMANTIC_TOP_K,
    "final_top_k": FINAL_TOP_K,
    "corpus_chunks": len(chunks),
    "faiss_vectors": index.ntotal,
    "source_phase": "Phase 6.5",
    "phase_6_5_selection_score": 0.9061,
    "phase_6_5_dev_mean_top1": 0.8825,
    "phase_6_5_dev_mean_top3": 0.9355,
    "phase_6_5_dev_mean_top5": 0.9518,
    "phase_6_5_holdout_mean_top1": 0.8194,
    "phase_6_5_holdout_mean_top3": 0.9167,
    "phase_6_5_holdout_mean_top5": 0.9167,
}

config_output = OUTPUT_DIR / "phase7_retrieval_config.json"

with open(config_output, "w", encoding="utf-8") as f:
    json.dump(
        retrieval_config,
        f,
        indent=2
    )

print("Saved:", config_output)

Saved: phase7_outputs\phase7_retrieval_config.json


Cell 19 — Test multiple queries

In [23]:
# ============================================================
# CELL 19: MULTI-QUERY RETRIEVAL TEST
# ============================================================

test_queries = [
    "What should I do if I have a workplace complaint?",
    "What are the company's core values?",
    "What benefits are available to employees?",
]

for i, query in enumerate(test_queries, start=1):

    start = time.perf_counter()

    results = retriever.retrieve(query)

    elapsed = time.perf_counter() - start

    print("\n" + "=" * 90)
    print(f"QUERY {i}")
    print("=" * 90)

    print(query)

    print(f"\nRetrieval time: {elapsed:.4f} seconds")

    for result in results:

        print(
            f"\nRank {result['final_rank']} "
            f"| Chunk {result['chunk_index']} "
            f"| Reranker {result['reranker_score']:.4f}"
        )

        print(
            result["text"][:300]
            .replace("\n", " ")
        )


QUERY 1
What should I do if I have a workplace complaint?

Retrieval time: 1.0077 seconds

Rank 1 | Chunk 20 | Reranker 1.1765
Clef is committed to creating a safe work environment that is free of threats to the health, safety, and wellbeing of the people who work here. That includes (but isn’t limited to) harassment, discrimination, violation of health and safety rules, and violence.  Clef has an open-door policy, so emplo

Rank 2 | Chunk 66 | Reranker 1.0634
It’s useful to take notes during the meeting, so that you can ask about how things have changed in the future. It’s also useful to take notes during the week -- if you notice something that you’d like to talk about, a note will make sure you address it in your next one on one.  If an employee is hav

Rank 3 | Chunk 21 | Reranker -2.7036
Currently, Clef is too small to have an internal group or department  that can independently respond to complaints, so if the founders are named in complaints, they will do their best to hold one